# 04 - PySpark 기초 (Docker 환경)

DataTalksClub data-engineering-zoomcamp **Module 6 – Batch Processing** 노트북을
Docker Jupyter 환경에 맞게 재구성한 버전입니다.

| 항목 | 값 |
|------|----|
| Spark | 3.5 (bitnami/spark) |
| Jupyter | jupyter/pyspark-notebook:spark-3.5.0 |
| 데이터 | FHVHV Trip Data 2021-01 |

## 1. SparkSession 생성

In [ ]:
import pyspark
from pyspark.sql import SparkSession

print(f'PySpark version: {pyspark.__version__}')

In [ ]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('zoomcamp-pyspark') \
    .config('spark.jars.ivy', '/tmp/.ivy2') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
spark

## 2. 데이터 다운로드

FHVHV(For-Hire Vehicle High Volume) 2021-01 데이터를 다운로드합니다.

> 파일 크기 약 ~130 MB (gzip 압축)

In [ ]:
!wget -q --show-progress \
    https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz \
    -O data/fhvhv_tripdata_2021-01.csv.gz

In [ ]:
!gzip -dkf data/fhvhv_tripdata_2021-01.csv.gz
!wc -l data/fhvhv_tripdata_2021-01.csv

In [ ]:
!head -n 5 data/fhvhv_tripdata_2021-01.csv

## 3. CSV 읽기 — 스키마 추론

In [ ]:
df = spark.read \
    .option('header', 'true') \
    .csv('data/fhvhv_tripdata_2021-01.csv')

df.printSchema()
print(f'Row count: {df.count():,}')

In [ ]:
df.show(5)

## 4. Pandas로 스키마 힌트 얻기

작은 샘플(1000행)을 Pandas로 읽어 dtype을 확인한 뒤,
Spark StructType으로 **명시적 스키마**를 정의합니다.

In [ ]:
!head -n 1001 data/fhvhv_tripdata_2021-01.csv > data/head.csv

In [ ]:
import pandas as pd

df_pandas = pd.read_csv('data/head.csv')
df_pandas.dtypes

In [ ]:
spark.createDataFrame(df_pandas).schema

## 5. 명시적 스키마 정의

- `Integer` = 4 bytes,  `Long` = 8 bytes
- LocationID → Integer로 충분
- datetime → TimestampType

In [ ]:
from pyspark.sql import types

schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

schema

## 6. 스키마 적용하여 다시 읽기

In [ ]:
df = spark.read \
    .option('header', 'true') \
    .schema(schema) \
    .csv('data/fhvhv_tripdata_2021-01.csv')

df.printSchema()

In [ ]:
df.show(5)

## 7. Repartition → Parquet 저장

24개 파티션으로 나눠 Parquet 포맷으로 저장합니다.

In [ ]:
df = df.repartition(24)
print(f'Partitions: {df.rdd.getNumPartitions()}')

In [ ]:
df.write.parquet('data/fhvhv/2021/01/', mode='overwrite')

In [ ]:
!ls -lh data/fhvhv/2021/01/ | head

## 8. Parquet 읽기 & 확인

In [ ]:
df = spark.read.parquet('data/fhvhv/2021/01/')
df.printSchema()
df.show(5)

## 9. 필터링 & 선택

`SELECT * FROM df WHERE hvfhs_license_num = 'HV0003'`

In [ ]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003') \
  .show(5)

## 10. UDF (User Defined Function)

In [ ]:
from pyspark.sql import functions as F

def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

# 테스트
crazy_stuff('B02884')

In [ ]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [ ]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

## 11. 정리

Spark 세션을 종료합니다.

In [ ]:
spark.stop()
print('Done!')